# Projet Machine Learning — Prédiction des Maladies Cardiaques
**IFOAD | Dr Arthur Sawadogo**

Dataset : Heart Disease UCI — https://archive.ics.uci.edu/dataset/45/heart+disease

## 0. Importation des bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    RocCurveDisplay, classification_report
)
import joblib

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 6)
print('✅ Bibliothèques importées avec succès')

## 1. Chargement des données

In [ ]:
# Chargement depuis le fichier CSV local
df = pd.read_csv('heart_disease_data.csv')

# Renommer la colonne cible si nécessaire (num -> target, binarisation)
if 'num' in df.columns and 'target' not in df.columns:
    df['target'] = (df['num'] > 0).astype(int)
    df = df.drop(columns=['num'])

# Renommer thalch -> thalach si nécessaire
if 'thalch' in df.columns:
    df = df.rename(columns={'thalch': 'thalach'})

print(f'Dimensions du dataset : {df.shape}')
print(f'Colonnes : {list(df.columns)}')
df.head()

## 2. Exploration des données (EDA)

In [ ]:
# Informations générales
print('=== Informations générales ===')
df.info()
print('\n=== Statistiques descriptives ===')
df.describe()

In [ ]:
# Valeurs manquantes
print('=== Valeurs manquantes ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'Aucune valeur manquante ✅')

In [ ]:
# Distribution de la variable cible
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

counts = df['target'].value_counts()
axes[0].pie(counts, labels=['Pas de maladie', 'Maladie cardiaque'],
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[0].set_title('Distribution de la variable cible')

sns.countplot(x='target', data=df, palette=['#2ecc71', '#e74c3c'], ax=axes[1])
axes[1].set_title('Nombre de patients par classe')
axes[1].set_xticklabels(['Pas de maladie (0)', 'Maladie (1)'])

plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### Question 1 — Distribution de l'âge

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['age'], kde=True, bins=20, color='steelblue', ax=axes[0])
axes[0].set_title('Distribution de l\'âge des individus')
axes[0].set_xlabel('Âge (années)')
axes[0].set_ylabel('Fréquence')

sns.boxplot(x='target', y='age', data=df,
            palette=['#2ecc71', '#e74c3c'], ax=axes[1])
axes[1].set_title('Âge selon la présence de maladie cardiaque')
axes[1].set_xticklabels(['Pas de maladie', 'Maladie'])
axes[1].set_xlabel('')
axes[1].set_ylabel('Âge (années)')

plt.suptitle('Question 1 : Distribution de l\'âge', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('q1_age_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Âge moyen global : {df['age'].mean():.1f} ans")
print(f"Âge moyen (sans maladie) : {df[df['target']==0]['age'].mean():.1f} ans")
print(f"Âge moyen (avec maladie) : {df[df['target']==1]['age'].mean():.1f} ans")

### Question 2 — Différence par sexe

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sex_target = df.groupby(['sex', 'target']).size().unstack()
sex_target.index = ['Femme', 'Homme']
sex_target.columns = ['Pas de maladie', 'Maladie']
sex_target.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'],
                edgecolor='white', rot=0)
axes[0].set_title('Maladie cardiaque par sexe')
axes[0].set_ylabel('Nombre de patients')
axes[0].legend(loc='upper right')

sex_pct = sex_target.div(sex_target.sum(axis=1), axis=0) * 100
sex_pct.plot(kind='bar', ax=axes[1], color=['#2ecc71', '#e74c3c'],
             edgecolor='white', rot=0)
axes[1].set_title('Pourcentage de maladie par sexe')
axes[1].set_ylabel('Pourcentage (%)')
axes[1].legend(loc='upper right')

plt.suptitle('Question 2 : Différence par sexe', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('q2_sex_disease.png', dpi=150, bbox_inches='tight')
plt.show()

### Question 3 — Type de douleur thoracique (cp) et maladie cardiaque

In [ ]:
cp_labels = {0: 'Angine typique', 1: 'Angine atypique',
             2: 'Douleur non-anginale', 3: 'Asymptomatique'}
df['cp_label'] = df['cp'].map(cp_labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(x='cp_label', hue='target', data=df,
              palette=['#2ecc71', '#e74c3c'], ax=axes[0])
axes[0].set_title('Type de douleur thoracique vs Maladie')
axes[0].set_xlabel('Type de douleur')
axes[0].tick_params(axis='x', rotation=20)
axes[0].legend(['Pas de maladie', 'Maladie'])

cp_pct = df.groupby('cp_label')['target'].mean() * 100
cp_pct.plot(kind='barh', color='#e74c3c', ax=axes[1])
axes[1].set_title('% maladie par type de douleur thoracique')
axes[1].set_xlabel('% patients avec maladie')

plt.suptitle('Question 3 : Douleur thoracique et maladie cardiaque',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('q3_chest_pain.png', dpi=150, bbox_inches='tight')
plt.show()

df.drop(columns=['cp_label'], inplace=True)

### Question 4 — Valeurs moyennes des indicateurs cliniques

In [ ]:
cols = ['trestbps', 'chol', 'thalach']
labels = ['Pression artérielle au repos\n(mm Hg)', 'Cholestérol\n(mg/dl)', 'Fréquence cardiaque max\n(bpm)']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, (col, label) in enumerate(zip(cols, labels)):
    sns.boxplot(x='target', y=col, data=df,
                palette=['#2ecc71', '#e74c3c'], ax=axes[i])
    axes[i].set_title(label)
    axes[i].set_xticklabels(['Pas de maladie', 'Maladie'])
    axes[i].set_xlabel('')

plt.suptitle('Question 4 : Valeurs moyennes des indicateurs cliniques',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('q4_clinical_indicators.png', dpi=150, bbox_inches='tight')
plt.show()

print(df.groupby('target')[cols].mean().rename(index={0:'Pas de maladie', 1:'Maladie'}).round(2))

### Question 5 — Glycémie à jeun (fbs) et maladie cardiaque

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

fbs_target = df.groupby(['fbs', 'target']).size().unstack()
fbs_target.index = ['fbs ≤ 120 mg/dl', 'fbs > 120 mg/dl']
fbs_target.columns = ['Pas de maladie', 'Maladie']
fbs_target.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'],
                edgecolor='white', rot=0)
axes[0].set_title('Glycémie à jeun vs Maladie cardiaque')
axes[0].set_ylabel('Nombre de patients')

fbs_pct = df.groupby('fbs')['target'].mean() * 100
fbs_pct.index = ['fbs ≤ 120', 'fbs > 120']
fbs_pct.plot(kind='bar', color=['#3498db', '#e74c3c'], ax=axes[1],
             edgecolor='white', rot=0)
axes[1].set_title('% maladie selon la glycémie à jeun')
axes[1].set_ylabel('% patients avec maladie')

plt.suptitle('Question 5 : Glycémie à jeun et maladie cardiaque',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('q5_fbs.png', dpi=150, bbox_inches='tight')
plt.show()

### Question 6 — Angine induite par l'exercice (exang) et maladie cardiaque

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

exang_target = df.groupby(['exang', 'target']).size().unstack()
exang_target.index = ['Sans angine', 'Avec angine']
exang_target.columns = ['Pas de maladie', 'Maladie']
exang_target.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'],
                  edgecolor='white', rot=0)
axes[0].set_title('Angine à l\'exercice vs Maladie')
axes[0].set_ylabel('Nombre de patients')

exang_pct = df.groupby('exang')['target'].mean() * 100
exang_pct.index = ['Sans angine', 'Avec angine']
exang_pct.plot(kind='bar', color=['#3498db', '#e74c3c'], ax=axes[1],
               edgecolor='white', rot=0)
axes[1].set_title('% maladie selon l\'angine à l\'exercice')
axes[1].set_ylabel('% patients avec maladie')

plt.suptitle('Question 6 : Angine à l\'exercice et maladie cardiaque',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('q6_exang.png', dpi=150, bbox_inches='tight')
plt.show()

### Matrice de corrélation

In [ ]:
plt.figure(figsize=(13, 9))
corr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.5, center=0, square=True)
plt.title('Matrice de corrélation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Préparation des données

In [ ]:
# Gestion des valeurs manquantes
df_clean = df.copy()
for col in df_clean.columns:
    if df_clean[col].isnull().sum() > 0:
        if df_clean[col].dtype == 'object':
            df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)
        else:
            df_clean[col].fillna(df_clean[col].median(), inplace=True)

# Séparation features / target
X = df_clean.drop(columns=['target'])
y = df_clean['target']

# Split train / test (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalisation
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train : {X_train.shape[0]} samples | Test : {X_test.shape[0]} samples')
print(f'Features : {list(X.columns)}')

## 4. Entraînement des 6 algorithmes de classification

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN':                 KNeighborsClassifier(n_neighbors=5),
    'SVM':                 SVC(kernel='rbf', probability=True, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'AdaBoost':            AdaBoostClassifier(n_estimators=100, random_state=42),
}

results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train_sc, y_train)
    y_pred = model.predict(X_test_sc)
    y_prob = model.predict_proba(X_test_sc)[:, 1]

    results.append({
        'Modèle':     name,
        'Accuracy':   round(accuracy_score(y_test, y_pred), 4),
        'Précision':  round(precision_score(y_test, y_pred), 4),
        'Rappel':     round(recall_score(y_test, y_pred), 4),
        'F1-Score':   round(f1_score(y_test, y_pred), 4),
        'AUC-ROC':    round(roc_auc_score(y_test, y_prob), 4),
    })
    trained_models[name] = model
    print(f'✅ {name} entraîné')

print('\nTous les modèles entraînés avec succès !')

## 5. Comparaison des résultats

In [ ]:
df_results = pd.DataFrame(results).set_index('Modèle')
df_results = df_results.sort_values('AUC-ROC', ascending=False)

print('=== Tableau comparatif des métriques ===')
display(df_results.style
    .background_gradient(cmap='RdYlGn', axis=0)
    .format('{:.4f}'))

In [ ]:
# Graphique comparatif des métriques
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

df_plot = df_results.reset_index()
metrics = ['Accuracy', 'Précision', 'Rappel', 'F1-Score', 'AUC-ROC']

df_melt = df_plot.melt(id_vars='Modèle', value_vars=metrics,
                       var_name='Métrique', value_name='Score')
sns.barplot(x='Modèle', y='Score', hue='Métrique', data=df_melt,
            ax=axes[0], palette='Set2')
axes[0].set_title('Comparaison des métriques par modèle')
axes[0].set_ylim(0, 1.1)
axes[0].tick_params(axis='x', rotation=25)
axes[0].legend(loc='lower right', fontsize=8)

# AUC-ROC uniquement
colors = sns.color_palette('Set2', len(df_results))
bars = axes[1].barh(df_results.index, df_results['AUC-ROC'], color=colors)
axes[1].set_title('AUC-ROC par modèle (du meilleur au moins bon)')
axes[1].set_xlim(0, 1.1)
for bar, val in zip(bars, df_results['AUC-ROC']):
    axes[1].text(val + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('models_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Courbes ROC de tous les modèles
fig, ax = plt.subplots(figsize=(10, 7))
colors_roc = sns.color_palette('Set2', len(models))

for (name, model), color in zip(trained_models.items(), colors_roc):
    RocCurveDisplay.from_estimator(model, X_test_sc, y_test,
                                   name=name, ax=ax, color=color)

ax.plot([0, 1], [0, 1], 'k--', label='Aléatoire (AUC=0.5)')
ax.set_title('Courbes ROC — Comparaison des 6 modèles', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Matrices de confusion (tous les modèles)
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, (name, model) in enumerate(trained_models.items()):
    y_pred = model.predict(X_test_sc)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Prédit: 0', 'Prédit: 1'],
                yticklabels=['Réel: 0', 'Réel: 1'])
    axes[i].set_title(f'{name}\nAcc={accuracy_score(y_test,y_pred):.3f}')

plt.suptitle('Matrices de confusion — 6 modèles', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Importance des features (Random Forest)

In [ ]:
rf_model = trained_models['Random Forest']
feat_imp = pd.Series(rf_model.feature_importances_, index=X.columns)
feat_imp = feat_imp.sort_values(ascending=True)

plt.figure(figsize=(10, 6))
colors = ['#e74c3c' if v >= feat_imp.quantile(0.75) else '#3498db' for v in feat_imp]
feat_imp.plot(kind='barh', color=colors)
plt.title('Importance des features — Random Forest', fontsize=14, fontweight='bold')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Sauvegarde du meilleur modèle

In [ ]:
best_model_name = df_results['AUC-ROC'].idxmax()
best_model = trained_models[best_model_name]

joblib.dump(best_model, 'heart_disease_pred_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

print(f'✅ Meilleur modèle : {best_model_name}')
print(f'   AUC-ROC : {df_results.loc[best_model_name, "AUC-ROC"]:.4f}')
print('✅ Modèle sauvegardé dans heart_disease_pred_model.pkl')
print('✅ Scaler sauvegardé dans scaler.pkl')

## 8. Conclusion

Dans ce projet, nous avons :
1. **Exploré** le dataset Heart Disease UCI à travers 6 questions analytiques
2. **Préparé** les données (nettoyage, normalisation, split train/test)
3. **Entraîné** 6 algorithmes de classification
4. **Comparé** leurs performances avec 5 métriques
5. **Sauvegardé** le meilleur modèle pour l'application Streamlit

Le modèle le plus performant selon l'AUC-ROC est utilisé dans l'application web Streamlit.